In [9]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    log_loss
)
from sklearn.utils import resample
import math

# -----------------------------
# Dataset configuration
# -----------------------------
INPUT_IMAGE = 256
BATCH_SIZE = 32
NUM_CLASSES = 4

datagen = ImageDataGenerator(rescale=1.0 / 255)

test_set = datagen.flow_from_directory(
    r"test",
    target_size=(INPUT_IMAGE, INPUT_IMAGE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

N_SAMPLES = test_set.samples
STEPS = math.ceil(N_SAMPLES / BATCH_SIZE)

# -----------------------------
# Prediction function
# -----------------------------
def get_predictions(model_path, generator):
    model = load_model(model_path)

    generator.reset()
    y_prob = model.predict(generator, steps=STEPS, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = generator.classes

    return y_true, y_pred, y_prob

# -----------------------------
# Metric computation
# -----------------------------
def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred) * 100
    loss = log_loss(y_true, y_prob)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    return {
        "accuracy": acc,
        "cce_loss": loss,
        "macro_precision": precision * 100,
        "macro_recall": recall * 100,
        "macro_f1": f1 * 100,
    }

# -----------------------------
# Bootstrap evaluation
# -----------------------------
def bootstrap_metrics(y_true, y_pred, y_prob, n_bootstraps=50, seed=42):
    rng = np.random.default_rng(seed)
    boot_results = []

    n = len(y_true)

    for _ in range(n_bootstraps):
        indices = rng.choice(n, size=n, replace=True)

        metrics = compute_metrics(
            y_true[indices],
            y_pred[indices],
            y_prob[indices],
        )
        boot_results.append(metrics)

    return pd.DataFrame(boot_results)

# -----------------------------
# Summary statistics
# -----------------------------
def summarize_bootstrap(df):
    summary = {}
    for metric in df.columns:
        values = df[metric].values
        summary[metric] = {
            "mean": np.mean(values),
            "std": np.std(values),
            "95_ci": [
                np.percentile(values, 2.5),
                np.percentile(values, 97.5),
            ],
        }
    return summary

# -----------------------------
# Full evaluation pipeline
# -----------------------------
def evaluate_model(model_path, generator, n_bootstraps=30):
    y_true, y_pred, y_prob = get_predictions(model_path, generator)
    df_boot = bootstrap_metrics(y_true, y_pred, y_prob, n_bootstraps)
    return summarize_bootstrap(df_boot)

# -----------------------------
# Model paths
# -----------------------------
model_paths = {
    "Custom-Built": r"Teacher_To_Student\CNNbasedTeacher_distil_T10.h5",
    "VGG16": r"VGG16_To_Student\VGG16Teacher_distil_T10.h5",
    "InceptionV3": r"InceptionV3_TO_Student\INCEPTIONV3_Teacher_distil2_T10_52724.h5",
    "MobileNetV2": r"MobileNetV2_To_Student\MobileNetV2Teacher_distil2_T10_52724.h5",
    "DenseNet121": r"DenseNet121_To_Student\DenseNet121Teacher_distil2_T10.h5",
    "No RKD": r"Teacher_To_Student\NewStudentAccuracy.h5",
}

# -----------------------------
# Run evaluation
# -----------------------------
results = {}
for name, path in model_paths.items():
    results[name] = evaluate_model(path, test_set, n_bootstraps=30)

# -----------------------------
# Print results (paper-ready)
# -----------------------------
for name, summ in results.items():
    acc = summ["accuracy"]
    loss = summ["cce_loss"]

    print(f"{name}:")
    print(
        f"  Accuracy: {acc['mean']:.2f}% ± {acc['std']:.2f}%, "
        f"95% CI [{acc['95_ci'][0]:.2f}%, {acc['95_ci'][1]:.2f}%]"
    )
    print(f"  CCE Loss: {loss['mean']:.4f} ± {loss['std']:.4f}")


Found 595 images belonging to 4 classes.
Custom-Built:
  Accuracy: 94.58% ± 0.78%, 95% CI [93.40%, 96.52%]
  CCE Loss: 0.1594 ± 0.0169
VGG16:
  Accuracy: 98.24% ± 0.56%, 95% CI [97.31%, 99.04%]
  CCE Loss: 0.0623 ± 0.0096
InceptionV3:
  Accuracy: 93.18% ± 1.02%, 95% CI [91.44%, 95.22%]
  CCE Loss: 0.1626 ± 0.0218
MobileNetV2:
  Accuracy: 87.82% ± 1.48%, 95% CI [85.91%, 91.00%]
  CCE Loss: 0.3246 ± 0.0367
DenseNet121:
  Accuracy: 89.88% ± 1.24%, 95% CI [87.67%, 92.15%]
  CCE Loss: 0.2419 ± 0.0237
No RKD:
  Accuracy: 77.08% ± 1.45%, 95% CI [74.63%, 79.13%]
  CCE Loss: 0.5908 ± 0.0250


In [10]:
rows = []

for model_name, summ in results.items():
    rows.append({
        "Model": model_name,
        "Accuracy (%)": f"{summ['accuracy']['mean']:.2f} ± {summ['accuracy']['std']:.2f}",
        "Accuracy 95% CI": f"[{summ['accuracy']['95_ci'][0]:.2f}, {summ['accuracy']['95_ci'][1]:.2f}]",
        "Macro F1 (%)": f"{summ['macro_f1']['mean']:.2f} ± {summ['macro_f1']['std']:.2f}",
        "CCE Loss": f"{summ['cce_loss']['mean']:.4f} ± {summ['cce_loss']['std']:.4f}",
    })

df_results = pd.DataFrame(rows)
df_results


,Model,Accuracy (%),Accuracy 95% CI,Macro F1 (%),CCE Loss
0,Custom-Built,94.58 ± 0.78,"[93.40, 96.52]",94.77 ± 0.76,0.1594 ± 0.0169
1,VGG16,98.24 ± 0.56,"[97.31, 99.04]",98.25 ± 0.54,0.0623 ± 0.0096
2,InceptionV3,93.18 ± 1.02,"[91.44, 95.22]",93.17 ± 1.03,0.1626 ± 0.0218
3,MobileNetV2,87.82 ± 1.48,"[85.91, 91.00]",87.96 ± 1.42,0.3246 ± 0.0367
4,DenseNet121,89.88 ± 1.24,"[87.67, 92.15]",89.77 ± 1.22,0.2419 ± 0.0237
5,No RKD,77.08 ± 1.45,"[74.63, 79.13]",76.78 ± 1.50,0.5908 ± 0.0250


In [1]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# =========================
# CONFIGURATION
# =========================
INPUT_IMAGE = 256
BATCH_SIZE = 32
NUM_CLASSES = 4
N_BOOTSTRAP = 1000
SEED = 42

# =========================
# MODEL PATHS
# =========================
model_paths = {
    "Custom-Built": r"saved_models\best_distilled_student_T10_seed1918.h5",
    "VGG16": r"VGG16_To_Student\VGG16Teacher_distil_T10.h5",
    "InceptionV3": r"InceptionV3_TO_Student\INCEPTIONV3_Teacher_distil2_T10_52724.h5",
    "MobileNetV2": r"MobileNetV2_To_Student\MobileNetV2Teacher_distil2_T10_52724.h5",
    "DenseNet121": r"DenseNet121_To_Student\DenseNet121Teacher_distil2_T10.h5",
    "No RKD": r"Teacher_To_Student\NewStudentAccuracy.h5",
}

# =========================
# LOAD TEST DATA
# =========================
datagen = ImageDataGenerator(rescale=1.0 / 255)

test_set = datagen.flow_from_directory(
    r"test",
    target_size=(INPUT_IMAGE, INPUT_IMAGE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

y_true = test_set.classes

# =========================
# BOOTSTRAP FUNCTION
# =========================
def bootstrap_metrics(y_true, y_pred, n_bootstrap=1000, seed=42):
    rng = np.random.default_rng(seed)

    acc_scores = []
    prec_scores = []
    rec_scores = []
    f1_scores = []

    for _ in range(n_bootstrap):
        indices = rng.integers(0, len(y_true), len(y_true))

        y_true_bs = y_true[indices]
        y_pred_bs = y_pred[indices]

        acc_scores.append(accuracy_score(y_true_bs, y_pred_bs))
        prec_scores.append(precision_score(y_true_bs, y_pred_bs, average="macro", zero_division=0))
        rec_scores.append(recall_score(y_true_bs, y_pred_bs, average="macro", zero_division=0))
        f1_scores.append(f1_score(y_true_bs, y_pred_bs, average="macro", zero_division=0))

    def summarize(scores):
        scores = np.array(scores)
        mean = scores.mean()
        std = scores.std()
        ci_low = np.percentile(scores, 2.5)
        ci_high = np.percentile(scores, 97.5)
        return mean, std, ci_low, ci_high

    return {
        "Accuracy": summarize(acc_scores),
        "Precision": summarize(prec_scores),
        "Recall": summarize(rec_scores),
        "F1-score": summarize(f1_scores),
    }

# =========================
# EVALUATE MODELS
# =========================
results = {}

for model_name, model_path in model_paths.items():
    print(f"\nEvaluating model: {model_name}")

    if not os.path.exists(model_path):
        print(f"Model not found: {model_path}")
        continue

    model = tf.keras.models.load_model(model_path)

    predictions = model.predict(test_set, verbose=0)
    y_pred = np.argmax(predictions, axis=1)

    # Point estimates
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    # Bootstrap statistics
    bootstrap_stats = bootstrap_metrics(
        y_true,
        y_pred,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED
    )

    results[model_name] = {
        "Point Accuracy": acc,
        "Point Precision": prec,
        "Point Recall": rec,
        "Point F1": f1,
        "Bootstrap": bootstrap_stats
    }

# =========================
# PRINT RESULTS (PAPER READY)
# =========================
print("\n================ FINAL RESULTS ================\n")

for model, res in results.items():
    print(f"Model: {model}")
    print(f"  Accuracy  : {res['Point Accuracy']:.4f}")
    print(f"  Precision : {res['Point Precision']:.4f}")
    print(f"  Recall    : {res['Point Recall']:.4f}")
    print(f"  F1-score  : {res['Point F1']:.4f}")

    for metric, (mean, std, ci_low, ci_high) in res["Bootstrap"].items():
        print(f"  {metric} (Mean ± Std): {mean:.4f} ± {std:.4f}")
        print(f"  {metric} 95% CI      : [{ci_low:.4f}, {ci_high:.4f}]")

    print("-" * 50)


c:\Users\My Pc\Desktop\Jisan\journal\review reponse applied soft computing\Review Progress file\R1C8\Rice-Leaf-Disease-Classification-using-Response-Based-Knowledge-Distillation\tf215_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(



Found 595 images belonging to 4 classes.

Evaluating model: Custom-Built



Evaluating model: VGG16

Evaluating model: InceptionV3

Evaluating model: MobileNetV2

Evaluating model: DenseNet121

Evaluating model: No RKD

================ FINAL RESULTS ================

Model: Custom-Built
  Accuracy  : 0.9731
  Precision : 0.9750
  Recall    : 0.9731
  F1-score  : 0.9738
  Accuracy (Mean ± Std): 0.9729 ± 0.0065
  Accuracy 95% CI      : [0.9597, 0.9849]
  Precision (Mean ± Std): 0.9749 ± 0.0060
  Precision 95% CI      : [0.9632, 0.9861]
  Recall (Mean ± Std): 0.9729 ± 0.0065
  Recall 95% CI      : [0.9600, 0.9850]
  F1-score (Mean ± Std): 0.9736 ± 0.0064
  F1-score 95% CI      : [0.9611, 0.9854]
--------------------------------------------------
Model: VGG16
  Accuracy  : 0.9815
  Precision : 0.9821
  Recall    : 0.9816
  F1-score  : 0.9816
  Accuracy (Mean ± Std): 0.9815 ± 0.0056
  Accuracy 95% CI      : [0.9697, 0.9916]
  Precision (Mean ± Std): 0.9820 ± 0.0055
  Precision 95% CI     

In [2]:
import pandas as pd

table_rows = []

for model, res in results.items():
    acc_mean, acc_std, acc_ci_low, acc_ci_high = res["Bootstrap"]["Accuracy"]

    row = {
        "Model": model,

        # Accuracy with uncertainty
        "Accuracy": res["Point Accuracy"],
        "Acc Mean": acc_mean,
        "Acc Std": acc_std,
        "Acc 95% CI": f"[{acc_ci_low:.4f}, {acc_ci_high:.4f}]",

        # Point estimates only
        "Precision": res["Point Precision"],
        "Recall": res["Point Recall"],
        "F1-score": res["Point F1"],
    }

    table_rows.append(row)

df_results = pd.DataFrame(table_rows)
df_results

,Model,Accuracy,Acc Mean,Acc Std,Acc 95% CI,Precision,Recall,F1-score
0,Custom-Built,0.973109,0.972941,0.006515,"[0.9597, 0.9849]",0.975032,0.973086,0.973792
1,VGG16,0.981513,0.981486,0.005628,"[0.9697, 0.9916]",0.982059,0.981597,0.981608
2,InceptionV3,0.932773,0.932375,0.010330,"[0.9126, 0.9529]",0.933690,0.933995,0.932456
3,MobileNetV2,0.882353,0.881829,0.013262,"[0.8555, 0.9076]",0.888796,0.883841,0.883295
4,DenseNet121,0.899160,0.899010,0.012775,"[0.8723, 0.9227]",0.908318,0.897519,0.897542
5,No RKD,0.778151,0.777850,0.017290,"[0.7429, 0.8084]",0.791125,0.783416,0.775703


In [3]:
import pandas as pd

models = list(results.keys())

table_data = {
    "Metric": [
        "Accuracy ↑",
        "95% CI",
        "Precision",
        "Recall",
        "F1-score",
    ]
}

for model in models:
    acc_mean, acc_std, acc_ci_low, acc_ci_high = results[model]["Bootstrap"]["Accuracy"]

    table_data[model] = [
        f"{acc_mean*100:.2f}% ± {acc_std*100:.2f}%",
        f"[{acc_ci_low*100:.2f}%, {acc_ci_high*100:.2f}%]",
        f"{results[model]['Point Precision']*100:.2f}%",
        f"{results[model]['Point Recall']*100:.2f}%",
        f"{results[model]['Point F1']*100:.2f}%",
    ]

df_comparison = pd.DataFrame(table_data)
df_comparison

,Metric,Custom-Built,VGG16,InceptionV3,MobileNetV2,DenseNet121,No RKD
0,Accuracy ↑,97.29% ± 0.65%,98.15% ± 0.56%,93.24% ± 1.03%,88.18% ± 1.33%,89.90% ± 1.28%,77.79% ± 1.73%
1,95% CI,"[95.97%, 98.49%]","[96.97%, 99.16%]","[91.26%, 95.29%]","[85.55%, 90.76%]","[87.23%, 92.27%]","[74.29%, 80.84%]"
2,Precision,97.50%,98.21%,93.37%,88.88%,90.83%,79.11%
3,Recall,97.31%,98.16%,93.40%,88.38%,89.75%,78.34%
4,F1-score,97.38%,98.16%,93.25%,88.33%,89.75%,77.57%


In [4]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# =========================
# CONFIGURATION
# =========================
INPUT_IMAGE = 256
BATCH_SIZE = 32
NUM_CLASSES = 4
N_BOOTSTRAP = 1000
SEED = 42

# =========================
# MODEL PATHS
# =========================
model_paths = {
    "Custom-Built": r"saved_models\best_distilled_student_T10_seed1918.h5",
    "VGG16": r"VGG16_To_Student\VGG16Teacher_distil_T10.h5",
    "InceptionV3": r"InceptionV3_TO_Student\INCEPTIONV3_Teacher_distil2_T10_52724.h5",
    "MobileNetV2": r"MobileNetV2_To_Student\MobileNetV2Teacher_distil2_T10_52724.h5",
    "DenseNet121": r"DenseNet121_To_Student\DenseNet121Teacher_distil2_T10.h5",
    "No RKD": r"Teacher_To_Student\NewStudentAccuracy.h5",
}

# =========================
# LOAD TEST DATA
# =========================
datagen = ImageDataGenerator(rescale=1.0 / 255)

test_set = datagen.flow_from_directory(
    r"test",
    target_size=(INPUT_IMAGE, INPUT_IMAGE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

y_true = test_set.classes
# Get one-hot encoded labels for cross-entropy calculation
y_true_one_hot = tf.keras.utils.to_categorical(y_true, num_classes=NUM_CLASSES)

# =========================
# BOOTSTRAP FUNCTION (MODIFIED TO INCLUDE CROSS-ENTROPY)
# =========================
def bootstrap_metrics(y_true, y_pred, y_true_one_hot, predictions, n_bootstrap=1000, seed=42):
    rng = np.random.default_rng(seed)

    acc_scores = []
    prec_scores = []
    rec_scores = []
    f1_scores = []
    ce_scores = []  # Cross-entropy scores

    for _ in range(n_bootstrap):
        indices = rng.integers(0, len(y_true), len(y_true))

        y_true_bs = y_true[indices]
        y_pred_bs = y_pred[indices]
        y_true_one_hot_bs = y_true_one_hot[indices]
        predictions_bs = predictions[indices]

        acc_scores.append(accuracy_score(y_true_bs, y_pred_bs))
        prec_scores.append(precision_score(y_true_bs, y_pred_bs, average="macro", zero_division=0))
        rec_scores.append(recall_score(y_true_bs, y_pred_bs, average="macro", zero_division=0))
        f1_scores.append(f1_score(y_true_bs, y_pred_bs, average="macro", zero_division=0))
        ce_scores.append(log_loss(y_true_one_hot_bs, predictions_bs))

    def summarize(scores):
        scores = np.array(scores)
        mean = scores.mean()
        std = scores.std()
        ci_low = np.percentile(scores, 2.5)
        ci_high = np.percentile(scores, 97.5)
        return mean, std, ci_low, ci_high

    return {
        "Accuracy": summarize(acc_scores),
        "Precision": summarize(prec_scores),
        "Recall": summarize(rec_scores),
        "F1-score": summarize(f1_scores),
        "Cross-Entropy": summarize(ce_scores),
    }

# =========================
# EVALUATE MODELS
# =========================
results = {}

for model_name, model_path in model_paths.items():
    print(f"\nEvaluating model: {model_name}")

    if not os.path.exists(model_path):
        print(f"Model not found: {model_path}")
        continue

    # Load model with compile=False to suppress warnings
    model = tf.keras.models.load_model(model_path, compile=False)
    
    # Compile the model (optional for inference)
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Get predictions (probabilities)
    predictions = model.predict(test_set, verbose=0)
    y_pred = np.argmax(predictions, axis=1)

    # Point estimates
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    
    # Calculate cross-entropy loss
    ce_loss = log_loss(y_true_one_hot, predictions)
    
    # Calculate confusion matrix for additional insights
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate per-class accuracy
    per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

    # Bootstrap statistics
    bootstrap_stats = bootstrap_metrics(
        y_true,
        y_pred,
        y_true_one_hot,
        predictions,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED
    )

    results[model_name] = {
        "Point Accuracy": acc,
        "Point Precision": prec,
        "Point Recall": rec,
        "Point F1": f1,
        "Point Cross-Entropy": ce_loss,
        "Per-Class Accuracy": per_class_accuracy,
        "Confusion Matrix": cm,
        "Bootstrap": bootstrap_stats
    }

# =========================
# PRINT RESULTS (PAPER READY)
# =========================
print("\n" + "=" * 60 + " FINAL RESULTS " + "=" * 60 + "\n")

for model, res in results.items():
    print(f"\nModel: {model}")
    print("-" * 40)
    print(f"Point Estimates:")
    print(f"  Accuracy       : {res['Point Accuracy']:.4f}")
    print(f"  Precision      : {res['Point Precision']:.4f}")
    print(f"  Recall         : {res['Point Recall']:.4f}")
    print(f"  F1-score       : {res['Point F1']:.4f}")
    print(f"  Cross-Entropy  : {res['Point Cross-Entropy']:.4f}")
    
    print(f"\nPer-Class Accuracy:")
    for i, acc in enumerate(res['Per-Class Accuracy']):
        print(f"  Class {i}: {acc:.4f}")
    
    print(f"\nBootstrap Statistics (Mean ± Std):")
    for metric, (mean, std, ci_low, ci_high) in res["Bootstrap"].items():
        print(f"  {metric:15s}: {mean:.4f} ± {std:.4f}")
        print(f"    {'95% CI':15s}: [{ci_low:.4f}, {ci_high:.4f}]")

# =========================
# ADDITIONAL ANALYSIS
# =========================
print("\n" + "=" * 60 + " MODEL COMPARISON " + "=" * 60 + "\n")

# Create a summary table
print("\nModel Performance Comparison (Ranked by F1-score):")
print("=" * 80)
print(f"{'Model':20s} {'Accuracy':10s} {'F1-score':10s} {'Cross-Entropy':15s}")
print("-" * 80)

# Sort models by F1-score
sorted_results = sorted(results.items(), key=lambda x: x[1]['Point F1'], reverse=True)

for model_name, res in sorted_results:
    print(f"{model_name:20s} {res['Point Accuracy']:.4f}      {res['Point F1']:.4f}      {res['Point Cross-Entropy']:.4f}")

print("\n" + "=" * 60 + " BEST MODEL " + "=" * 60 + "\n")
best_model = sorted_results[0]
print(f"Best Performing Model: {best_model[0]}")
print(f"Accuracy: {best_model[1]['Point Accuracy']:.4f}")
print(f"F1-score: {best_model[1]['Point F1']:.4f}")
print(f"Cross-Entropy: {best_model[1]['Point Cross-Entropy']:.4f}")

# =========================
# SAVE RESULTS TO FILE
# =========================
import json
import pickle

# Save detailed results
results_summary = {}
for model_name, res in results.items():
    results_summary[model_name] = {
        "Accuracy": float(res["Point Accuracy"]),
        "Precision": float(res["Point Precision"]),
        "Recall": float(res["Point Recall"]),
        "F1": float(res["Point F1"]),
        "CrossEntropy": float(res["Point Cross-Entropy"]),
        "PerClassAccuracy": [float(x) for x in res["Per-Class Accuracy"]],
        "Bootstrap": {
            metric: {
                "mean": float(stats[0]),
                "std": float(stats[1]),
                "ci_low": float(stats[2]),
                "ci_high": float(stats[3])
            }
            for metric, stats in res["Bootstrap"].items()
        }
    }

# Save as JSON
with open("model_evaluation_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)

# Save as pickle for later analysis
with open("model_evaluation_results.pkl", "wb") as f:
    pickle.dump(results, f)

print("\nResults saved to 'model_evaluation_results.json' and 'model_evaluation_results.pkl'")

Found 595 images belonging to 4 classes.

Evaluating model: Custom-Built


Evaluating model: VGG16

Evaluating model: InceptionV3

Evaluating model: MobileNetV2

Evaluating model: DenseNet121

Evaluating model: No RKD

============================================================ FINAL RESULTS ============================================================


Model: Custom-Built
----------------------------------------
Point Estimates:
  Accuracy       : 0.9731
  Precision      : 0.9750
  Recall         : 0.9731
  F1-score       : 0.9738
  Cross-Entropy  : 0.0736

Per-Class Accuracy:
  Class 0: 0.9874
  Class 1: 0.9375
  Class 2: 0.9750
  Class 3: 0.9924

Bootstrap Statistics (Mean ± Std):
  Accuracy       : 0.9729 ± 0.0065
    95% CI         : [0.9597, 0.9849]
  Precision      : 0.9749 ± 0.0060
    95% CI         : [0.9632, 0.9861]
  Recall         : 0.9729 ± 0.0065
    95% CI         : [0.9600, 0.9850]
  F1-score       : 0.9736 ± 0.0064
    95% CI         : [0.9611, 0.9854]
  Cross-Entrop